### Baseline Model ChatBundestag

In [32]:
import os

# data handling & viz
import pandas as pd
import matplotlib.pyplot as plt

# language preprocessing
import re #regex
from wordcloud import WordCloud
import spacy # DE stopwords

# langchain packages
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.faiss import DistanceStrategy

from langchain_core.documents import Document
from langchain_core.runnables import RunnablePassthrough #may not be needed due to QA removal
from langchain_core.runnables import RunnableLambda
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.embeddings import Embeddings
#from langchain_core.output_parsers import StrOutputParser appears to be buggy

from langchain_groq import ChatGroq
from langchain_classic import hub # alternative to 'from langchain import hub', because this gave errors


from pydantic import BaseModel, ValidationError
from typing import Optional, Literal, List

import json

# environment variables
from dotenv import load_dotenv
load_dotenv()
import warnings
warnings.filterwarnings('ignore')

# instantiate ChatGroq with llama
llm = ChatGroq(
    model= "llama-3.1-8b-instant",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2
)


# refs for vector db handling
vector_db_name = "debates_lp19_Mini_LM-L6-v2"
vector_db_path = f"vector_databases/vector_db_{vector_db_name}"

In [17]:
# cleaned and reduced data set
df_exp_debates = pd.read_csv("data/debates_lp19.csv")

In [18]:
df_exp_debates.shape

(26869, 11)

### Chunking

In [19]:
# all-MiniLM-L6-v2: ~256 tokens ≈ 200 chars safe upper bound
# Switch these up when you move to multilingual-e5-small (512 tokens ≈ 400 chars)

TINY_DOC_THRESHOLD   = 50    # e.g. interjections
SMALL_DOC_THRESHOLD  = 400   # short statements = 1 chunk
MEDIUM_DOC_THRESHOLD = 1500  # medium speeches = paragraph-level split
# anything above: full recursive split with overlap

CHUNK_SIZE_MEDIUM = 500      # characters
CHUNK_SIZE_LARGE  = 400      # characters
CHUNK_OVERLAP     = 50       # roughly 10%

BATCH_SIZE = 500             # reduce to 250 if RAM issues


In [20]:
# split text
# todo: hard speech split to mark beginning and end of MPs speech (document)
def get_splitter(chunk_size):
    return RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ".", ";", ","], 
       
        
    )

paragraph_splitter = get_splitter(CHUNK_SIZE_MEDIUM)
token_splitter     = get_splitter(CHUNK_SIZE_LARGE)


In [21]:
# dynamic chunking (document-aware)
def chunk_single_document(doc: Document, doc_idx: int) -> list[Document]:
    """
    Applies dynamic chunking based on document length.
    Never merges across documents.

    Size buckets (characters):
        tiny   < 50     : 1 chunk, flagged (low semantic signal)
        small  < 400    : 1 chunk
        medium < 1500   : paragraph-level split
        large  ≥ 1500   : recursive token-level split with overlap
    """
    text = doc.page_content.strip()

    if not text:
        return []

    length = len(text)

    # Determine strategy
    if length < TINY_DOC_THRESHOLD:
        strategy = "tiny"
        splits = [doc]

    elif length < SMALL_DOC_THRESHOLD:
        strategy = "small"
        splits = [doc]

    elif length < MEDIUM_DOC_THRESHOLD:
        strategy = "medium"
        splits = paragraph_splitter.split_documents([doc])

    else:
        strategy = "large"
        splits = token_splitter.split_documents([doc])

    # attach metadata to every chunk
    total = len(splits)
    for i, chunk in enumerate(splits):
        chunk.metadata.update({
            "chunk_id":          f"doc{doc_idx}_chunk{i}",
            "chunk_index":       i,
            "total_chunks":      total,
            "is_full_document":  total == 1,
            "chunking_strategy": strategy,
            "doc_char_length":   length,
        })

    return splits


In [22]:
# batch pipeline
def df_to_document_batches(df, batch_size=BATCH_SIZE):
    """Generator: yields batches of Documents without loading full df into memory."""
    for start in range(0, len(df), batch_size):
        batch = df.iloc[start:start + batch_size]
        yield [
            Document(
                page_content=row['text'],
                metadata={
                    'row_index':        i,
                    'speaker_name':     row['speech_identification_ent'],
                    'date':             row['date'],
                    'legislative_period': row['period'],
                    'session':          row['session'],
                    'role':             row['Role'],
                    'governing_party':  row['governing_Party'],
                    'party':            row['Party'],
                }
            )
            for i, row in batch.iterrows()
        ]

In [23]:
# run chunking pipeline
def run_chunking_pipeline(df, batch_size=BATCH_SIZE) -> list[Document]:
    """
    Full batched chunking pipeline.
    Tracks strategy distribution so you can inspect and tune thresholds.
    """
    all_chunks = []
    strategy_counts = {"tiny": 0, "small": 0, "medium": 0, "large": 0}
    total_docs = 0

    for batch_idx, doc_batch in enumerate(df_to_document_batches(df, batch_size)):
        for local_idx, doc in enumerate(doc_batch):
            global_idx = batch_idx * batch_size + local_idx
            chunks = chunk_single_document(doc, global_idx)
            all_chunks.extend(chunks)

            if chunks:
                strategy = chunks[0].metadata["chunking_strategy"]
                strategy_counts[strategy] += 1

        total_docs += len(doc_batch)
        print(f"Batch {batch_idx + 1}: {total_docs} docs processed → "
              f"{len(all_chunks)} total chunks")

    print(f"\nChunking strategy distribution:\n{strategy_counts}")
    print(f"Total chunks: {len(all_chunks)} from {total_docs} documents")
    return all_chunks

In [24]:
# chunking
chunks = run_chunking_pipeline(df_exp_debates)

Batch 1: 500 docs processed → 6254 total chunks
Batch 2: 1000 docs processed → 12357 total chunks
Batch 3: 1500 docs processed → 18494 total chunks
Batch 4: 2000 docs processed → 25388 total chunks
Batch 5: 2500 docs processed → 33217 total chunks
Batch 6: 3000 docs processed → 40820 total chunks
Batch 7: 3500 docs processed → 46900 total chunks
Batch 8: 4000 docs processed → 55160 total chunks
Batch 9: 4500 docs processed → 63264 total chunks
Batch 10: 5000 docs processed → 69676 total chunks
Batch 11: 5500 docs processed → 76077 total chunks
Batch 12: 6000 docs processed → 84460 total chunks
Batch 13: 6500 docs processed → 90807 total chunks
Batch 14: 7000 docs processed → 97241 total chunks
Batch 15: 7500 docs processed → 103525 total chunks
Batch 16: 8000 docs processed → 110116 total chunks
Batch 17: 8500 docs processed → 115941 total chunks
Batch 18: 9000 docs processed → 122754 total chunks
Batch 19: 9500 docs processed → 128457 total chunks
Batch 20: 10000 docs processed → 1350

In [25]:
# display results of chunking function
print(f"number of chunks created: {len(chunks)}")

number of chunks created: 356390


### Embedding

In [45]:
# instantiate embedding model 
embedding = HuggingFaceEmbeddings(
        model_name='intfloat/multilingual-e5-small', 
        # cloud upgrade path: intfloat/multilingual-e5-base or deepset/gbert-base or BAAI/bge-m3
        model_kwargs={'device': 'mps'}, 
        # cloud: 'mps' -> 'cuda'.
        encode_kwargs={"normalize_embeddings": True,
                      "prompt": "passage: "  # adds prefix automatically at indexing time
                      }
    )

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 3ef3043f-6fbd-4fb2-b6d7-874286a45be1)')' thrown while requesting HEAD https://huggingface.co/intfloat/multilingual-e5-small/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].


In [46]:
# prefix handling for embedding model
class E5QueryWrapper(Embeddings):
    """
    Wraps HuggingFaceEmbeddings to add the required 'query: ' prefix
    for multilingual-e5 models at retrieval time only.
    """
    def __init__(self, base_embedding: Embeddings):
        self.base = base_embedding

    def embed_query(self, text: str) -> List[float]:
        return self.base.embed_query(f"query: {text}")

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        # prefix 'passage: ' already handled in encode_kwargs at indexing time
        return self.base.embed_documents(texts)

embedding_wrapped = E5QueryWrapper(embedding)

In [47]:
# function to create and save vector store 
def create_and_store(chunks,vector_db_path,embedding):
    """
    this function creates a vector store from chunks and saves it locally
    """
    # create the vector store 
    vectorstore = FAISS.from_documents(
        documents=chunks,
        embedding=embedding, # parameter name is singular!
        distance_strategy=DistanceStrategy.COSINE  # or DistanceStrategy.DOT or DistanceStrategy.L2 
    )
    
     # save vector store locally
    vectorstore.save_local(vector_db_path)

    return vectorstore

In [48]:
# implement retrieval from FAISS db

def retrieve_from_vector_db(vector_db_path,embedding):
    """
    this function splits out a retriever object from a local vector store
    """
    
    vectorstore = FAISS.load_local(
        folder_path=vector_db_path,
        embeddings=embedding, # parameter name is plural!
        allow_dangerous_deserialization=True,
        distance_strategy=DistanceStrategy.COSINE  # or DistanceStrategy.DOT or DistanceStrategy.L2 
    )
    retriever = vectorstore.as_retriever(
        search_kwargs={
            'k':15, # k nearest
        }  
    )
    
    return retriever,vectorstore


In [49]:
# check if vector store exists. if no: creates vector store
if not os.path.exists(vector_db_path):
        print("Vector DB not found. Creating and embedding chunks.")
        vectorstore=create_and_store(chunks=chunks, vector_db_path=vector_db_path, embedding=embedding_wrapped)
        print(f"Vector DB save to {vector_db_path}")
else:
    print(f"Vector DB found at {vector_db_path}. Skipping embedding.")

Vector DB found at vector_databases/vector_db_debates_lp19. Skipping embedding.


In [50]:
# load the retriever and index
retriever,vectorstore = retrieve_from_vector_db(vector_db_path=vector_db_path, embedding=embedding_wrapped)

#type(retriever),type(vectorstore)

### Prompt

In [51]:
# prompt
prompt = ChatPromptTemplate.from_template(
    """
    Du bist ein Assistent für Argumentationsanalyse, spezialisiert auf parlamentarische Debatten des Deutschen Bundestags.
    
    Deine Aufgabe ist es, die Argumentationsstruktur aus dem untenstehenden Redeauszug zu extrahieren,
    wobei du die Frage des Nutzers als thematischen Fokus verwendest.
    
    Definitionen:
    - claim: die zentrale politische Position oder der Vorschlag, für den oder gegen den argumentiert wird
    - supporting_premises: Aussagen, Fakten oder Gründe, die den claim stützen
    - opposing_premises: Gegenargumente, Einwände oder Positionen, die abgelehnt werden
    
    Regeln:
    - Verwende NUR Informationen aus dem Kontext, keine externen Kenntnisse
    - Zitiere oder paraphrasiere eng aus dem Text, nichts erfinden
    - Fokussiere die Extraktion auf die Frage des Nutzers, ignoriere thematisch irrelevante Textteile
    - Wenn kein klarer claim erkennbar ist, setze "claim" auf null
    - Wenn keine opposing_premises vorhanden sind, gib eine leere Liste zurück
    - Gib NUR valides JSON zurück, keine Erklärung, keine Einleitung
    
    Beispiel Frage: "Welche Argumente werden für die Energiewende genannt?"
    
    Beispiel Input:
    "Wir müssen die Energiewende beschleunigen. Der Klimawandel bedroht unsere Lebensgrundlagen. 
    Erneuerbare Energien schaffen zudem neue Arbeitsplätze. Die Opposition behauptet, dies sei zu teuer, 
    doch die Kosten des Nichtstuns sind weitaus höher."
    
    Beispiel Output:
    {{
        "claim": "Die Energiewende muss beschleunigt werden",
        "supporting_premises": [
            "Der Klimawandel bedroht unsere Lebensgrundlagen",
            "Erneuerbare Energien schaffen neue Arbeitsplätze"
        ],
        "opposing_premises": [
            "Die Energiewende sei zu teuer"
        ],
        "confidence": "high",
        "reasoning": "Der Sprecher formuliert eine klare politische Forderung und stützt diese mit zwei expliziten Begründungen. Ein Gegenargument wird genannt und direkt entkräftet. Die Extraktion fokussiert auf Energiewende-Argumente gemäß der Nutzerfrage."
    }}
    
    Kontext:
    {context}
    
    Frage:
    {question}
    """)

In [52]:
# Pydantic schema for output
class ArgumentStructure(BaseModel):
    claim: Optional[str]
    supporting_premises: list[str]
    opposing_premises: list[str]
    confidence: Literal["high", "medium", "low"] 
    reasoning: str 

In [53]:
# check and format LLM output: according to Pydantic schema
def parse_llm_output(raw_output: str) -> dict:
    """
    Parses and validates JSON output from the LLM.
    Returns parsed dict on success, error dict on failure.
    """
    try:
        # strip markdown code fences if model wraps output in ```json ... ```
        cleaned = re.sub(r"```json|```", "", raw_output).strip()
        
        # parse JSON
        parsed = json.loads(cleaned)
        
        # validate against schema
        validated = ArgumentStructure(**parsed)
        
        return {"status": "ok", "data": validated.model_dump()}
    
    except json.JSONDecodeError as e:
        return {"status": "json_error", "raw": raw_output, "error": str(e)}
    
    except ValidationError as e:
        return {"status": "validation_error", "raw": raw_output, "error": str(e)}

In [54]:
# runnable RAG pipeline (LCEL)
def connect_chains(retriever):
    """
    Connects retriever, prompt and llm into a RAG chain for argument extraction.
    User question both drives semantic search and focuses the extraction.
    """
    rag_chain = (
        RunnableLambda(lambda x: {"context": retriever.invoke(x), "question": x})
        | prompt
        | llm
        | RunnableLambda(lambda msg: parse_llm_output(msg.content))
    )
    return rag_chain

In [55]:
rag_chain = connect_chains(retriever)

In [56]:
def chat_with_rag(chain):
    print("Willkommen im ChatBundestag 🏛️. Gib eine Frage zu den Bundestagsdebatten ein.\nSchreibe 'exit', um den Chat zu beenden.\n")
    while True:
        user_input = input("Du: ")
        if user_input.lower() in ["exit", "quit"]:
            print("Chat wird beendet. Auf Wiedersehen!")
            break
        try:
            result = chain.invoke(user_input)
            
            if result["status"] == "ok":
                data = result["data"]
                print(f"\n📌 Claim: {data['claim']}")
                print(f"✅ Stützende Prämissen:")
                for p in data['supporting_premises']:
                    print(f"   - {p}")
                print(f"❌ Gegenargumente:")
                for p in data['opposing_premises']:
                    print(f"   - {p}")
                print(f"🎯 Konfidenz: {data['confidence']}\n")
            else:
                print(f"⚠️ Parsing fehlgeschlagen ({result['status']}): {result['error']}\n")
                print(f"Rohausgabe: {result['raw']}\n")  # log raw for debugging
        
        except Exception as e:
            print(f"Fehler: {e}\n")

In [ ]:
# run chat
chat_with_rag(rag_chain)

### error log
- always identify speaker, party and date
- don't generate output if query is unclear/ ill-formulated / lacking essential components
- prompt does not take into account {reasoning} for now
- how is the topic introduced for which to retrieve claim/premises ? Query should filter the database preemptively - so query + context in prompt!
- '\xa0–' as part of output

### query log

-- Was sind die Argumente der Grünen für das Klimaschutzgesetz? (OK output for V0)


- Wie steht die FDP zur Mietpreisbremse? (identical output for supporting / opposing)
- Wie steht die Regierung zur Mietpreisbremse? (error 413)
- Wie steht die CDU zum Brexit? (json_error, Parsing fehlgeschlagen)
- Was sagt Angela Merkel zur Covid-19 Situation? (wrong extraction, query does not respect pre-defined format)
- Was sind Angela Merkels Argumente für Corona-Hilfen? (no output whatsoever)

- Finde Merkels Argumente zur Maskenaffäre (characters out of place)
- Wie steht Angela Merkel zum Thema Maskenpflicht? (output empty)
- Welche Argumente werden für den Brexit genannt? (output: claim, rest: empty)
- Welche Argumente werden gegen den Brexit genannt? (output: claim, rest: empty)
- Was sind die Argumente der AfD gegen das Klimaschutzgesetz? (conflicting premises)
